# 🚨 Конспект: Винятки та Обробка Помилок у Python

> Урок 10 — повний конспект: що таке виняток, синтаксичні vs runtime помилки,
> механізм try/except/else/finally, raise, поширення помилок, EAFP vs LBYL.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Ментальна модель | Виняток як «контрольований збій» |
| 2 | SyntaxError vs RuntimeError | Коли програма падає і чому |
| 3 | Шість головних винятків | ValueError, TypeError, ZeroDivisionError, IndexError, KeyError, FileNotFoundError |
| 4 | try / except / else / finally | Механізм перехоплення |
| 5 | Небезпека «голого» except | «Підгузковий» антипатерн |
| 6 | raise — захисне програмування | Генерація власних винятків |
| 7 | Поширення помилок (Call Stack) | Stack Unwinding |
| 8 | EAFP vs LBYL | Філософія Python |
| 9 | Практичні вправи | Завдання для самостійної роботи |
| 10 | Підсумок | 5 ключових висновків |

---

## 🧠 Частина 1: Ментальна модель — Виняток як «Контрольований збій»

У програмуванні помилки бувають різними.

**Синтаксичні помилки (SyntaxError)** виникають, коли код граматично неправильний
(наприклад, забута дужка або двокрапка). Інтерпретатор Python не може навіть прочитати
такий код — програма не запуститься взагалі.

**Помилки часу виконання (Runtime Errors / Exceptions)** виникають, коли синтаксично
правильний код намагається виконати неможливу операцію під час роботи програми:
поділити на нуль, відкрити неіснуючий файл, звернутися до відсутнього ключа словника.

### Що робить Python, коли виникає runtime помилка?

Python не просто «падає». Він використовує механізм **контрольованого збою**:
створює **об'єкт винятку (Exception Object)**.

```
┌─────────────────────────────────────────────────────┐
│  Нормальний потік виконання                         │
│                                                     │
│  рядок 1 ✓ → рядок 2 ✓ → рядок 3 💥 ВИНЯТОК       │
│                                    ↓                │
│                         Запускається «сигнальна     │
│                         ракета» вгору по стеку      │
│                                    ↓                │
│                 Перший блок except, що збігається   │
│                 з типом → перехоплює та обробляє    │
│                                                     │
│                 Якщо ніхто не перехопить → КРАХ     │
│                 програми + Traceback                │
└─────────────────────────────────────────────────────┘
```

> **Аналогія:** уявіть, що функція «запускає сигнальну ракету» вгору.
> Якщо ви передбачили таку ситуацію — можна «перехопити» ракету і відновити роботу.
> Якщо ні — програма аварійно завершується і виводить трасування стека (Traceback).

---

## ⚡ Частина 2: SyntaxError vs RuntimeError

### Синтаксична помилка — програма НЕ запускається

In [ ]:
# SyntaxError: забута закриваюча дужка
# Python відмовиться навіть починати виконання
print("Hello"
print("World")

### Runtime Error — програма запускається, але падає під час виконання

In [ ]:
# Перший рядок ВИКОНАЄТЬСЯ, третій — НІ
print("Початок програми")   # ✓ виконується
result = 10 / 0              # 💥 ZeroDivisionError — зупинка тут
print("Кінець програми")    # ✗ ніколи не виконається

| Тип | Коли виникає | Чи запускається програма? | Приклад |
|-----|-------------|--------------------------|---------|
| `SyntaxError` | При читанні коду | ❌ Ні | Забута `:` або `(` |
| RuntimeError | Під час виконання | ✅ Так, але падає пізніше | `1/0`, `int("abc")` |

---

## 🎯 Частина 3: Шість головних винятків початківця

| Виняток | Коли виникає | Приклад коду |
|---------|-------------|--------------|
| `ValueError` | Правильний тип, але некоректне значення | `int("привіт")` |
| `TypeError` | Несумісні типи даних в операції | `"Оцінка: " + 5` |
| `ZeroDivisionError` | Ділення на нуль | `10 / 0` |
| `IndexError` | Індекс списку виходить за межі | `my_list[99]` (3 елементи) |
| `KeyError` | Ключ не існує у словнику | `my_dict["age"]` |
| `FileNotFoundError` | Файл не знайдено на диску | `open("ghost.txt")` |

In [ ]:
# Демонстрація кожного з шести винятків
# Розкоментуйте потрібний рядок і запустіть

# ValueError:
# int("привіт")

# TypeError:
# "Оцінка: " + 5

# ZeroDivisionError:
# 10 / 0

# IndexError:
# lst = [1, 2, 3]; print(lst[99])

# KeyError:
# d = {"name": "Alice"}; print(d["age"])

# FileNotFoundError:
# open("non_existent_file.txt")

print("Розкоментуйте один рядок вище і запустіть, щоб побачити виняток.")

---

## 🛡️ Частина 4: try / except / else / finally

Щоб перехопити виняток, «ризиковий» код поміщається у блок `try`.

### Схема роботи механізму

```
try:
    # ризикований код
    # (якщо тут виникне помилка — одразу стрибаємо до except)
except ТипПомилки:
    # код відновлення — виконується ТІЛЬКИ якщо помилка збіглася з типом
else:
    # виконується ТІЛЬКИ якщо try пройшов БЕЗ помилок
finally:
    # виконується ЗАВЖДИ — і при успіху, і при помилці
    # (ідеально для закриття файлів, з'єднань з БД)
```

### Повний приклад з усіма чотирма блоками

In [ ]:
def safe_divide():
    while True:
        try:
            # Ризиковані операції
            num_str = input("Введіть дільник для числа 100: ")
            divisor = int(num_str)       # може викликати ValueError
            result = 100 / divisor       # може викликати ZeroDivisionError

        except ValueError:
            # Спрацює, якщо введено "abc", "pizza" тощо
            print("❌ Це не число! Спробуйте ще раз.")

        except ZeroDivisionError:
            # Спрацює, якщо введено 0
            print("❌ На нуль ділити не можна!")

        else:
            # Виконується ТІЛЬКИ якщо try успішний (без помилок)
            print(f"✅ Успіх! 100 / {divisor} = {result:.4f}")
            break  # виходимо з циклу

        finally:
            # Виконується ЗАВЖДИ після кожної спроби
            print("--- Спроба завершена ---")

# safe_divide()  # розкоментуйте для запуску

### Порядок виконання — ключові правила

1. `try` виконується рядок за рядком. Як тільки виникає виняток — виконання **негайно** зупиняється і стрибає до `except`.
2. `except` блоки перевіряються **згори донизу** — спрацьовує перший збіг типу.
3. `else` виконується **тільки** якщо `try` пройшов без помилок.
4. `finally` виконується **завжди** — навіть якщо програма «падає» або виконується `return`.

In [ ]:
# Спостерігаємо за порядком виконання
try:
    print("try: рядок 1")
    x = int("не_число")   # 💥 ValueError тут
    print("try: рядок 3 — НЕ виконається")
except ValueError:
    print("except: перехоплено ValueError")
except ZeroDivisionError:
    print("except: цей блок НЕ перевіряється — перший вже збігся")
else:
    print("else: НЕ виконається — була помилка")
finally:
    print("finally: виконується ЗАВЖДИ")

---

## ☣️ Частина 5: Небезпека «голого» except — Антипатерн «Підгузок»

### Що таке «голий» except?

In [ ]:
# ПОГАНО: голий except без вказання типу
try:
    number = int("10")
    print(numbor)          # опечатка! — NameError
except:                    # НЕБЕЗПЕЧНО: ковтає ВСЕ
    print("Щось пішло не так при конвертації...")

### Чому це небезпечно?

Голий `except:` (або `except Exception:`) поглинає **абсолютно все**:

| Що поглинається | Наслідок |
|----------------|----------|
| `NameError` (опечатка в імені змінної) | Приховує баги в коді |
| `KeyboardInterrupt` | Користувач не може зупинити програму через Ctrl+C |
| `SystemExit` | Програма не може нормально завершитись |
| `MemoryError` | Критична системна помилка маскується |

> **Правило:** Завжди вказуйте конкретний тип винятку, який ви очікуєте.

In [ ]:
# ПРАВИЛЬНО: явно вказуємо тип
try:
    number = int("10")
    print(numbor)          # тепер NameError правомірно «впаде» — ми його не ховаємо
except ValueError:
    print("Помилка конвертації рядка в число.")
# NameError вийде назовні і покаже реальний баг!

---

## 🎯 Частина 6: raise — Захисне програмування

Ми можемо **самостійно генерувати** винятки за допомогою `raise`,
щоб захистити функції від некоректних вхідних даних.

Це називається **defensive programming** — програма явно відмовляється
продовжувати роботу зі «сміттєвими» даними, замість того щоб дозволити
помилці «заразити» систему глибше.

In [ ]:
def calculate_age(birth_year: int, current_year: int) -> int:
    if birth_year > current_year:
        # Явно «запускаємо ракету» вгору по стеку
        raise ValueError(f"Рік народження ({birth_year}) не може бути у майбутньому!")
    if birth_year < 1900:
        raise ValueError("Рік народження не може бути раніше 1900.")
    return current_year - birth_year


# Функція-викликач перехоплює і вирішує, що показати користувачу
try:
    age = calculate_age(2050, 2024)
    print(f"Вік: {age}")
except ValueError as e:   # 'as e' — зберігаємо об'єкт винятку зі своїм повідомленням
    print(f"❌ Помилка розрахунку: {e}")

In [ ]:
# Практичний приклад: функція доступу до даних
def get_user_record(user_id: int) -> str:
    users = {1: "Alice", 2: "Bob", 3: "Charlie"}

    if user_id < 0:
        raise ValueError("ID користувача не може бути від'ємним!")

    # Якщо ключа немає — Python сам згенерує KeyError
    return users[user_id]


def main():
    for test_id in [-5, 99, 2]:
        try:
            user = get_user_record(test_id)
            print(f"✅ Знайдено: {user}")
        except ValueError as e:
            print(f"❌ Помилка вхідних даних: {e}")
        except KeyError:
            print(f"❌ Користувача з ID={test_id} не знайдено.")

main()

---

## 📡 Частина 7: Поширення помилок (Stack Unwinding)

Що відбувається, якщо функція не перехоплює виняток?
Помилка **піднімається (спливає) по стеку викликів** до функції-викликача.
Якщо ніхто не перехопить — програма «падає».

```
function_a() ← має try/except
    └── function_b() ← немає try/except
            └── function_c() ← 💥 ZeroDivisionError тут

Стек «розкручується» (unwinds):
  C не перехоплює → кидає вгору до B
  B не перехоплює → кидає вгору до A
  A перехоплює! ✅ → програма виживає
```

In [ ]:
def function_c():
    print("  C: виконується")
    return 10 / 0              # 💥 виняток тут

def function_b():
    print(" B: виконується, викликає C")
    function_c()
    print(" B: цей рядок НЕ виконається")

def function_a():
    print("A: виконується, викликає B")
    try:
        function_b()
    except ZeroDivisionError:
        print("A: перехопив ZeroDivisionError з глибини стека!")

function_a()
print("Програма продовжує роботу після перехоплення ✅")

> **Ключова ідея:** Не потрібно ставити `try/except` у КОЖНУ функцію.
> Достатньо розмістити обробник там, де ви знаєте, **що саме** робити з помилкою.

---

## 🐍 Частина 8: EAFP vs LBYL — Філософія Python

### LBYL — Look Before You Leap (Перевір перед стрибком)
> Стиль з мов C, Java: перевіряй все через `if` перед виконанням дії.

In [ ]:
# LBYL — перевіряємо вручну перед кожною дією
import os

filename = "data.txt"

if os.path.exists(filename):             # перевірка 1
    if os.access(filename, os.R_OK):     # перевірка 2
        with open(filename) as f:
            content = f.read()
    else:
        print("Немає прав читання")
else:
    print("Файл не знайдено")

# Проблема: між перевіркою і відкриттям файл може зникнути (race condition!)

### EAFP — Easier to Ask Forgiveness than Permission (Легше вибачитись, ніж питати дозволу)
> Стиль Python: просто виконуй дію і обробляй виняток, якщо щось пішло не так.

In [ ]:
# EAFP — Python-стиль: просто пробуємо
try:
    with open("data.txt") as f:
        content = f.read()
except FileNotFoundError:
    print("Файл не знайдено")
except PermissionError:
    print("Немає прав читання")

# Переваги EAFP:
# ✅ Немає race condition
# ✅ Чистіший код (happy path не захаращений if-ами)
# ✅ try/except оптимізовано: майже нульова вартість якщо помилки немає

| | LBYL | EAFP |
|-|------|------|
| Стиль | Перевіряй все заздалегідь через `if` | Спробуй і обробляй виняток |
| Мови | C, Java, Go | Python, Ruby |
| Проблема | Race conditions, перевантаженість `if`-ами | Потрібно знати можливі типи винятків |
| Швидкість | Перевірки завжди виконуються | `try` без помилки — майже безкоштовний |

**Python явно рекомендує EAFP** — це стандартний «пітонічний» стиль.

---

## 💪 Частина 9: Практичні вправи

### Вправа 1: Цикл вводу з обробкою помилок

In [ ]:
# Завдання: Запитуйте рік народження, поки користувач не введе коректне число
# Підказка: while True + try/except ValueError + break

# YOUR CODE HERE
while True:
    try:
        year = int(input("Введіть рік народження: "))
        print(f"✅ Рік прийнято: {year}")
        break
    except ValueError:
        print("❌ Введіть ціле число (наприклад: 1995)")

### Вправа 2: Безпечний доступ до словника та ділення

In [ ]:
# Завдання: Система цін з багами
# prices = {'apple': 20, 'banana': 0, 'cherry': 50}  ← ціна банана = 0 (баг!)
#
# 1. Запитати у користувача назву фрукта
# 2. Отримати ціну (KeyError якщо немає)
# 3. Запитати бюджет (ValueError якщо введено не число)
# 4. Порахувати кількість: budget / price (ZeroDivisionError якщо ціна 0)
# 5. else: вивести скільки фруктів можна купити

prices = {'apple': 20, 'banana': 0, 'cherry': 50}
budget = 100  # можна зробити input()

fruit = input("Введіть назву фрукта: ")

try:
    price = prices[fruit]           # KeyError якщо немає
    amount = budget / price         # ZeroDivisionError якщо ціна 0
except KeyError:
    print(f"❌ Товару '{fruit}' немає в асортименті.")
except ZeroDivisionError:
    print(f"❌ Помилка ціни в системі: ціна '{fruit}' = 0.")
else:
    print(f"✅ За бюджет {budget}₴ можна купити {amount:.1f} шт. '{fruit}'")

### Вправа 3: Повний калькулятор

In [ ]:
# Завдання: Функція calculate(a, b) з обробкою всіх помилок
# Catch: TypeError (якщо передали рядок), ZeroDivisionError
# else: вивести результат якщо все ок

def calculate(a, b):
    return a / b

test_cases = [
    (10, 2),        # ✅ успіх
    (10, 0),        # ZeroDivisionError
    ("10", 2),      # TypeError
]

for a, b in test_cases:
    try:
        result = calculate(a, b)
    except ZeroDivisionError:
        print(f"❌ {a} / {b} → ZeroDivisionError: ділення на нуль")
    except TypeError:
        print(f"❌ {a} / {b} → TypeError: несумісні типи")
    else:
        print(f"✅ {a} / {b} = {result}")

### Вправа 4: Дебагінг — знайди та виправ баги

In [ ]:
# Баг 1: «Підгузковий» антипатерн приховує реальну помилку
# Що тут насправді пішло не так? Знайдіть і виправте.

my_list = [1, 2, 3]
try:
    my_list.apend(4)    # опечатка: apend замість append
except:
    print("Щось пішло не так.")

# Виправлення: замініть bare except на конкретний тип, або
# виправте опечатку в назві методу

In [ ]:
# Баг 2: Неправильний тип винятку в except
user_data = {"id": 101}
try:
    print(user_data["name"])
except IndexError:              # ← ПОМИЛКА: словник кидає KeyError, не IndexError!
    print("Дані не знайдено!")

# Виправте: замініть IndexError на правильний тип

In [ ]:
# Баг 3: Порядок except має значення
# Запустіть, подивіться що відбувається, поясніть чому

try:
    int("abc")
except Exception:           # дуже широкий тип — перехоплює ВСЕ
    print("Exception спрацював")
except ValueError:          # ніколи не досягається!
    print("ValueError спрацював")

# Правило: ставте специфічніші типи ВИЩЕ, загальніші — НИЖЧЕ

---

## 📝 Частина 10: Підсумок

### 5 ключових висновків

1. **Виняток = контрольований збій.** Коли Python не може виконати операцію,
   він не падає blindly — він створює об'єкт винятку і «запускає ракету» вгору по стеку.

2. **SyntaxError ≠ RuntimeError.** Синтаксична помилка зупиняє програму ще до запуску.
   Рантайм-помилка виникає під час виконання.

3. **try/except/else/finally:**
   - `try` — ризикований код
   - `except` — відновлення при конкретній помилці
   - `else` — тільки якщо try успішний
   - `finally` — завжди (очищення ресурсів)

4. **Ніколи не пишіть голий `except:`** — «підгузковий антипатерн» ховає реальні баги.
   Завжди вказуйте конкретний тип: `except ValueError:`.

5. **EAFP > LBYL.** Python-стиль — просто спробувати і обробити виняток,
   а не перевіряти все через `if` заздалегідь.

### Шпаргалка

```python
try:
    result = risky_operation()
except ValueError as e:
    print(f"Помилка значення: {e}")
except (KeyError, IndexError):    # можна об'єднати типи
    print("Дані не знайдено")
except ZeroDivisionError:
    print("Ділення на нуль")
else:
    print(f"Успіх: {result}")     # тільки якщо try ок
finally:
    cleanup()                     # завжди

# Генерація власного винятку
def validate(x):
    if x < 0:
        raise ValueError("x не може бути від'ємним")
```